# 04. Sentiment Model Training & Comparison

This notebook builds the core machine learning sentiment classifier. We convert star ratings into discrete sentiment categories, construct a TF-IDF feature matrix, train multiple algorithms, and serialize the best model to disk.

### Model Comparison:
1. Logistic Regression
2. Naive Bayes (MultinomialNB)
3. Linear Support Vector Machine (LinearSVC)


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('../processed/cleaned_reviews.csv')
df.head(2)


,product_id,product_name,category,review_title,review_text,rating,review_date,verified_purchase,full_review,cleaned_text,processed_tokens
0,AMZ-B000PHHY8O,Amazon Grocery Item (B000PHHY),Grocery & Gourmet,Retro Candy Gift Box,I bought this at Christmas time for my much ol...,5,2012-05-17,True,Retro Candy Gift Box I bought this at Christma...,retro candy gift box i bought this at christma...,retro candy gift box bought christmas time muc...
1,AMZ-B0029NTQ1K,Amazon Grocery Item (B0029NTQ),Grocery & Gourmet,Your cats will crave it fortnightly!,"Actually, if your cats are like mine, they wil...",5,2010-12-16,True,"Your cats will crave it fortnightly! Actually,...",your cats will crave it fortnightly actually i...,cats crave fortnightly actually cats like mine...


## Step 1: Feature Extraction (TF-IDF Vectorization)


In [3]:
X = df['processed_tokens'].fillna('')
y = df['rating'].apply(lambda r: 'negative' if r <= 2 else ('neutral' if r == 3 else 'positive'))

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_vec = vectorizer.fit_transform(X)
print('TF-IDF Matrix Shape:', X_vec.shape)


TF-IDF Matrix Shape: (10000, 1000)


## Step 2: Train / Test Stratified Split


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}')


Training samples: 7500, Testing samples: 2500


## Step 3: Model Training & Evaluation


In [5]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=200, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(C=1.0, random_state=42)
}

results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = (clf, acc, preds)
    print(f'=== {name} ===')
    print(f'Accuracy: {acc * 100:.2f}%')
    print(classification_report(y_test, preds, zero_division=0))


=== Logistic Regression ===
Accuracy: 85.52%
              precision    recall  f1-score   support

    negative       0.79      0.54      0.64       353
     neutral       0.50      0.11      0.18       183
    positive       0.87      0.98      0.92      1964

    accuracy                           0.86      2500
   macro avg       0.72      0.54      0.58      2500
weighted avg       0.83      0.86      0.83      2500

=== Multinomial Naive Bayes ===
Accuracy: 82.56%
              precision    recall  f1-score   support

    negative       0.93      0.29      0.44       353
     neutral       0.00      0.00      0.00       183
    positive       0.82      1.00      0.90      1964

    accuracy                           0.83      2500
   macro avg       0.58      0.43      0.45      2500
weighted avg       0.78      0.83      0.77      2500

=== Linear SVM ===
Accuracy: 85.12%
              precision    recall  f1-score   support

    negative       0.68      0.61      0.65       353

## Step 4: Serialize Best Model and Vectorizer to Disk


In [6]:
best_model_name = 'Logistic Regression'
best_model = results[best_model_name][0]

os.makedirs('../models', exist_ok=True)
model_path = '../models/sentiment_model.pkl'
vectorizer_path = '../models/vectorizer.pkl'

joblib.dump(best_model, model_path)
joblib.dump(vectorizer, vectorizer_path)

print(f'[+] Successfully saved trained model to: {model_path}')
print(f'[+] Successfully saved vectorizer to: {vectorizer_path}')


[+] Successfully saved trained model to: ../models/sentiment_model.pkl
[+] Successfully saved vectorizer to: ../models/vectorizer.pkl


### Conclusion:
- Logistic Regression achieved high multi-class precision and recall on sentiment classification.
- Artifacts `sentiment_model.pkl` and `vectorizer.pkl` are exported to `data/models/` for Express backend consumption.
